# Hybrid Retrieval Test
**Three signals → RRF fusion**
- Signal 1: KG traversal (NetworkX graph)
- Signal 2: Dense vector search (Qdrant)
- Signal 3: BM25 keyword search

Run cells top to bottom. Change `QUERY` in the last section to test different questions.

In [1]:
import sys, os
# Notebook lives at src/notebooks/ — one level up is src/
sys.path.insert(0, os.path.abspath('..'))

import re
import numpy as np
from pathlib import Path
from rank_bm25 import BM25Okapi

from config.config import KGConfig, AdvancedConfig
from data.chunk_loader import load_preprocessed_chunks
from indexing.kg_store import KGStore
from indexing.kg_retriever import KGRetriever
from indexing.embedder import TextEmbedder
from indexing.vector_database import QdrantVectorDB

KG_CFG  = KGConfig()
VEC_CFG = AdvancedConfig()
print('Config loaded')

/opt/homebrew/Caskroom/miniconda/base/envs/dat540/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Config loaded


## 1 — Load the Knowledge Graph

In [2]:
graph, kg_chunks = KGStore.load(KG_CFG.KG_GRAPH_FILE)
kg_retriever = KGRetriever(graph=graph, chunks=kg_chunks, hops=2)

print(f'Nodes : {graph.number_of_nodes():,}')
print(f'Edges : {graph.number_of_edges():,}')
print(f'Chunks: {len(kg_chunks):,}')

Nodes : 77,421
Edges : 73,011
Chunks: 12,747


## 2 — Build BM25 Index

In [3]:
def tokenize(text):
    return re.findall(r'\w+', text.lower())

chunks = load_preprocessed_chunks(Path(KG_CFG.PREPROCESSED_CHUNKS_FILE))
bm25 = BM25Okapi([tokenize(c['text']) for c in chunks])
print(f'BM25 index built over {len(chunks):,} chunks')

BM25 index built over 12,747 chunks


## 3 — Connect to Qdrant + Embedder

In [4]:

# The config default points to a collection that doesn't exist locally.
# Override to the populated one (19,888 text points).
VEC_CFG.VECTOR_DB_COLLECTION = "advanced_fixed_size"

embedder  = TextEmbedder(VEC_CFG.EMBEDDING_MODEL)
vector_db = QdrantVectorDB(VEC_CFG)
print(f'Qdrant collection : {VEC_CFG.VECTOR_DB_COLLECTION}')
print(f'Documents in index: {vector_db.count_documents():,}')


/Users/bebs/.cache/huggingface/modules/transformers_modules/jinaai/jina-clip-implementation/39e6a55ae971b59bea6e44675d237c99762e7ee2/modeling_clip.py:137: UserWarning: Flash attention requires CUDA, disabling
  warnings.warn('Flash attention requires CUDA, disabling')
/Users/bebs/.cache/huggingface/modules/transformers_modules/jinaai/jina-clip-implementation/39e6a55ae971b59bea6e44675d237c99762e7ee2/modeling_clip.py:172: UserWarning: xFormers requires CUDA, disabling
  warnings.warn('xFormers requires CUDA, disabling')


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.



[OK] Embedding model loaded. Dimension: 1024
Initializing local Qdrant at /Users/bebs/Documents/School/DAT 560/DAT560project/src/local_qdrant...


/Users/bebs/Documents/School/DAT 560/DAT560project/src/indexing/vector_database.py:94: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <advanced_sliding_window> contains 25791 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client = QdrantClient(path=local_path) # Initialize a local Qdrant instance that stores data on disk at the specified path (data will persist across runs)



[OK] Local Qdrant initialized
Qdrant collection : advanced_fixed_size
Documents in index: 19,888


## 4 — RRF Fusion Helper

In [5]:
RRF_K = 60

def rrf_fuse(ranked_lists, top_k=5):
    """
    ranked_lists: list of lists, each list is [{id, text, metadata, ...}, ...]
                  ordered best-first.
    Returns the top_k results fused by Reciprocal Rank Fusion.
    """
    scores = {}
    docs   = {}

    for ranked in ranked_lists:
        for rank, doc in enumerate(ranked):
            uid = doc.get('uid') or doc.get('id') or doc['text'][:60]
            scores[uid] = scores.get(uid, 0) + 1 / (RRF_K + rank)
            docs[uid]   = doc

    top_uids = sorted(scores, key=scores.get, reverse=True)[:top_k]
    return [{'rrf_score': scores[u], **docs[u]} for u in top_uids]

## 5 — Run a Query
Change `QUERY` and `TOP_K` as needed.

In [6]:
QUERY = "What is the relationship between ISIS and terrorism?"
TOP_K = 5
N_CANDIDATES = TOP_K * 10  # fetch more per signal before fusing

In [7]:
# ── Signal 1: KG traversal ───────────────────────────────────────────────────
kg_results = kg_retriever.retrieve(QUERY, top_k=N_CANDIDATES)
# normalise shape: add uid key for RRF
for r in kg_results:
    r['uid'] = r['metadata']['uid']

# ── Signal 2: Dense (Qdrant) ─────────────────────────────────────────────────
query_emb    = embedder.embed_query(QUERY)
dense_results = vector_db.retrieve(query_emb, top_k=N_CANDIDATES, allowed_types=['text'])
for r in dense_results:
    r['uid'] = str(r['id'])

# ── Signal 3: BM25 ───────────────────────────────────────────────────────────
bm25_scores  = bm25.get_scores(tokenize(QUERY))
bm25_top_ids = np.argsort(bm25_scores)[::-1][:N_CANDIDATES].tolist()
bm25_results = []
for idx in bm25_top_ids:
    c = chunks[idx]
    bm25_results.append({
        'uid'   : f"{c['pdf_name']}::{c['chunk_id']}",
        'text'  : c['text'],
        'score' : float(bm25_scores[idx]),
        'metadata': {'pdf_name': c['pdf_name'], 'chunk_id': c['chunk_id']},
    })

print(f'KG results   : {len(kg_results)}')
print(f'Dense results: {len(dense_results)}')
print(f'BM25 results : {len(bm25_results)}')

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


KG results   : 50
Dense results: 50
BM25 results : 50


## 6 — Inspect Each Signal Individually

In [8]:
def show(label, results, n=3):
    print(f'\n── {label} ──')
    for i, r in enumerate(results[:n]):
        meta = r.get('metadata') or r.get('payload') or {}
        pdf  = meta.get('pdf_name', '?')
        cid  = meta.get('chunk_id', '?')
        text = r.get('text','')[:120].replace('\n',' ')
        print(f'  [{i+1}] {pdf} | chunk {cid}')
        print(f'       {text}...')

show('KG traversal', kg_results)
show('Dense (Qdrant)', dense_results)
show('BM25', bm25_results)


── KG traversal ──
  [1] 2303.08559v2.pdf | chunk 8
       [Document: 2303.08559v2.pdf] s to enhance few-shot IE tasks. 2 Related Work 2.1 LLMs for Information Extraction...
  [2] NETFLIX_2015_10K.pdf | chunk 137
       [Document: NETFLIX_2015_10K.pdf] s at end of period                  | 4,904                                            ...
  [3] 2310.05634v2.pdf | chunk 19
       [Document: 2310.05634v2.pdf] The generation component effectively prompt the LLMs with the retrieved knowledge graphs (K...

── Dense (Qdrant) ──
  [1] 12-15-15-ISIS-and-terrorism-release-final.pdf | chunk 51
       [Document: 12-15-15-ISIS-and-terrorism-release-final.pdf] The analysis in this report is based on telephone interviews c...
  [2] 12-15-15-ISIS-and-terrorism-release-final.pdf | chunk 56
       [Document: 12-15-15-ISIS-and-terrorism-release-final.pdf] ints  | Sample sizes and sampling errors for other subgroups a...
  [3] 12-15-15-ISIS-and-terrorism-release-final.pdf | chunk 30
       [Document:

## 7 — Fuse with RRF

In [9]:
fused = rrf_fuse([kg_results, dense_results, bm25_results], top_k=TOP_K)

print(f'\n── Fused top-{TOP_K} (RRF) ──')
for i, r in enumerate(fused):
    meta = r.get('metadata') or r.get('payload') or {}
    pdf  = meta.get('pdf_name', '?')
    cid  = meta.get('chunk_id', '?')
    text = r.get('text','')[:150].replace('\n',' ')
    print(f'  [{i+1}] rrf={r["rrf_score"]:.4f} | {pdf} | chunk {cid}')
    print(f'       {text}...')
    print()


── Fused top-5 (RRF) ──
  [1] rrf=0.0167 | 2303.08559v2.pdf | chunk 8
       [Document: 2303.08559v2.pdf] s to enhance few-shot IE tasks. 2 Related Work 2.1 LLMs for Information Extraction...

  [2] rrf=0.0167 | 12-15-15-ISIS-and-terrorism-release-final.pdf | chunk 51
       [Document: 12-15-15-ISIS-and-terrorism-release-final.pdf] The analysis in this report is based on telephone interviews conducted December 8-13, 2015 a...

  [3] rrf=0.0167 | 12-15-15-ISIS-and-terrorism-release-final.pdf | chunk 0
       [Document: 12-15-15-ISIS-and-terrorism-release-final.pdf] FOR RELEASE DECEMBER 15, 2015 Views of Government's Handling of Terrorism Fall to Post-9/11 ...

  [4] rrf=0.0164 | NETFLIX_2015_10K.pdf | chunk 137
       [Document: NETFLIX_2015_10K.pdf] s at end of period                  | 4,904                                                         | 5,767          ...

  [5] rrf=0.0164 | 12-15-15-ISIS-and-terrorism-release-final.pdf | chunk 56
       [Document: 12-15-15-ISIS-and-terro